In [ ]:
from google.colab import files
uploaded = files.upload()


Saving INDIA_AQI_CLEANED (1).csv to INDIA_AQI_CLEANED (1).csv


In [ ]:
import os
os.rename(
    "/content/INDIA_AQI_CLEANED (1).csv",
    "/content/india_aqi.csv"
)
print(" File renamed successfully!")

 File renamed successfully!


In [ ]:

DATA_PATH = r"C:\Users\Priya\Downloads\INDIA_AQI_CLEANED (1).csv"


DATA_PATH = "/content/india_aqi.csv"

In [ ]:
import pandas as pd
df = pd.read_csv("/content/india_aqi.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))

Shape: (842015, 63)
Columns: ['City', 'State', 'Latitude', 'Longitude', 'Datetime', 'Year', 'Month', 'Day', 'Hour', 'Day_of_Week', 'Day_Name', 'Week_of_Year', 'Is_Weekend', 'Quarter', 'Season', 'Time_of_Day', 'Temp_2m_C', 'Humidity_Percent', 'Dew_Point_C', 'Humidity_Category', 'Wind_Speed_10m_kmh', 'Wind_Dir_10m', 'Wind_Gusts_kmh', 'Wind_Category', 'Wind_Stagnation', 'Precipitation_mm', 'Rain_mm', 'Is_Raining', 'Heavy_Rain', 'Pressure_MSL_hPa', 'Surface_Pressure_hPa', 'Solar_Radiation_Wm2', 'Direct_Radiation_Wm2', 'Diffuse_Radiation_Wm2', 'Cloud_Cover_Percent', 'Cloud_Low_Percent', 'Cloud_Mid_Percent', 'Cloud_High_Percent', 'Is_Daytime', 'Sunshine_Seconds', 'PM2_5_ugm3', 'PM10_ugm3', 'PM_Ratio', 'CO_ugm3', 'NO2_ugm3', 'SO2_ugm3', 'O3_ugm3', 'Dust_ugm3', 'AOD', 'US_AQI', 'US_AQI_PM25', 'US_AQI_PM10', 'US_AQI_NO2', 'US_AQI_O3', 'US_AQI_CO', 'EU_AQI', 'EU_AQI_PM25', 'EU_AQI_PM10', 'AQI_Category', 'PM25_Category_India', 'Temp_Inversion', 'Festival_Period', 'Crop_Burning_Season']
       Cit

In [ ]:
import pandas as pd
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Cities:", sorted(df["City"].unique()))
print("Columns:", df.columns.tolist())

Shape: (842015, 63)
Cities: ['Agartala', 'Ahmedabad', 'Aizawl', 'Bengaluru', 'Bhopal', 'Bhubaneswar', 'Chandigarh', 'Chennai', 'Dehradun', 'Delhi', 'Gangtok', 'Gurugram', 'Guwahati', 'Hyderabad', 'Imphal', 'Itanagar', 'Jaipur', 'Kohima', 'Kolkata', 'Lucknow', 'Mumbai', 'Panaji', 'Patna', 'Raipur', 'Ranchi', 'Shillong', 'Shimla', 'Thiruvananthapuram', 'Visakhapatnam']
Columns: ['City', 'State', 'Latitude', 'Longitude', 'Datetime', 'Year', 'Month', 'Day', 'Hour', 'Day_of_Week', 'Day_Name', 'Week_of_Year', 'Is_Weekend', 'Quarter', 'Season', 'Time_of_Day', 'Temp_2m_C', 'Humidity_Percent', 'Dew_Point_C', 'Humidity_Category', 'Wind_Speed_10m_kmh', 'Wind_Dir_10m', 'Wind_Gusts_kmh', 'Wind_Category', 'Wind_Stagnation', 'Precipitation_mm', 'Rain_mm', 'Is_Raining', 'Heavy_Rain', 'Pressure_MSL_hPa', 'Surface_Pressure_hPa', 'Solar_Radiation_Wm2', 'Direct_Radiation_Wm2', 'Diffuse_Radiation_Wm2', 'Cloud_Cover_Percent', 'Cloud_Low_Percent', 'Cloud_Mid_Percent', 'Cloud_High_Percent', 'Is_Daytime', 'Sun

In [ ]:
# ============================================================
# STEP 1: Mount Drive + Install Libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install tensorflow scikit-learn pandas numpy joblib -q

# ============================================================
# STEP 2: Imports
# ============================================================
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    Input, Flatten, RepeatVector,
    Permute, Multiply, Activation, Lambda,
    BatchNormalization, Conv1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# ============================================================
# STEP 3: CONFIG — Apna path yahan set karo
# ============================================================
DATA_PATH  = "/content/drive/MyDrive/india_aqi.csv"  # ← apna Drive path
MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN    = 48
EPOCHS     = 120
BATCH_SIZE = 32

FEATURES = [
    "US_AQI", "PM2_5_ugm3", "PM10_ugm3", "Temp_2m_C", "Humidity_Percent",
    "Wind_Speed_10m_kmh", "Surface_Pressure_hPa", "Solar_Radiation_Wm2", "Rain_mm",
]

os.makedirs(MODELS_DIR, exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)
print("✅ Setup done!")

# ============================================================
# STEP 4: Feature Engineering
# ============================================================
def add_features(df):
    df = df.copy()
    df["hour_sin"]  = np.sin(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["Datetime"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["Datetime"].dt.month / 12)
    df["dow_sin"]   = np.sin(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    for lag in [1, 2, 3, 6, 12, 24, 48]:
        df[f"AQI_lag{lag}"] = df["US_AQI"].shift(lag)
    for window in [3, 6, 12, 24, 48]:
        df[f"AQI_roll{window}_mean"] = df["US_AQI"].rolling(window, min_periods=1).mean()
        df[f"AQI_roll{window}_std"]  = df["US_AQI"].rolling(window, min_periods=1).std().fillna(0)
        df[f"AQI_roll{window}_max"]  = df["US_AQI"].rolling(window, min_periods=1).max()
    for diff in [1, 3, 6, 24]:
        df[f"AQI_diff{diff}"] = df["US_AQI"].diff(diff).fillna(0)
    df["PM_ratio"]        = (df["PM2_5_ugm3"] / (df["PM10_ugm3"] + 1e-6)).clip(0, 5)
    df["Heat_index"]      = df["Temp_2m_C"] * df["Humidity_Percent"] / 100
    df["Wind_dilution"]   = df["US_AQI"] / (df["Wind_Speed_10m_kmh"] + 1e-6)
    df["is_morning_rush"] = df["Datetime"].dt.hour.between(7, 10).astype(int)
    df["is_evening_rush"] = df["Datetime"].dt.hour.between(17, 20).astype(int)
    df["is_night"]        = df["Datetime"].dt.hour.between(22, 5).astype(int)
    df["is_weekend"]      = (df["Datetime"].dt.dayofweek >= 5).astype(int)
    return df

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i: i + seq_len])
        y.append(data[i + seq_len, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = Lambda(lambda t: K.sum(t, axis=1))(ctx)
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

def inverse_aqi(scaler, scaled_vals, n_features):
    dummy = np.zeros((len(scaled_vals), n_features))
    dummy[:, 0] = np.array(scaled_vals).flatten()
    return scaler.inverse_transform(dummy)[:, 0]

print("✅ Functions ready!")

# ============================================================
# STEP 5: Train Each City
# ============================================================
def train_city(city_df, city):
    print(f"\n{'='*50}")
    print(f"  Training: {city}  ({len(city_df)} rows)")
    print(f"{'='*50}")

    city_df = city_df.sort_values("Datetime").reset_index(drop=True)
    for col in FEATURES:
        if col in city_df.columns:
            city_df[col] = city_df[col].ffill().bfill().fillna(city_df[col].mean())

    city_df  = add_features(city_df)
    eng_cols = [
        "hour_sin","hour_cos","month_sin","month_cos","dow_sin","dow_cos",
        "AQI_lag1","AQI_lag2","AQI_lag3","AQI_lag6","AQI_lag12","AQI_lag24","AQI_lag48",
        "AQI_roll3_mean","AQI_roll6_mean","AQI_roll12_mean","AQI_roll24_mean","AQI_roll48_mean",
        "AQI_roll3_std","AQI_roll6_std","AQI_roll24_std",
        "AQI_roll3_max","AQI_roll6_max","AQI_roll24_max",
        "AQI_diff1","AQI_diff3","AQI_diff6","AQI_diff24",
        "PM_ratio","Heat_index","Wind_dilution",
        "is_morning_rush","is_evening_rush","is_night","is_weekend",
    ]
    all_feats = FEATURES + [c for c in eng_cols if c in city_df.columns]
    city_df   = city_df[all_feats].ffill().bfill().fillna(0)

    if len(city_df) < SEQ_LEN + 300:
        print(f"  ⚠️ Not enough data — skipping.")
        return None

    values = city_df.values.astype(np.float32)
    n_feat = values.shape[1]
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    X, y   = make_sequences(scaled, SEQ_LEN)

    n     = len(X)
    t_end = int(n * 0.80)
    v_end = int(n * 0.90)
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]
    print(f"  Shapes → Train:{len(X_tr)} Val:{len(X_v)} Test:{len(X_te)} Features:{n_feat}")

    model     = build_model(SEQ_LEN, n_feat)
    safe_name = city.replace(" ", "_")
    ckpt      = os.path.join(MODELS_DIR, f"{safe_name}_ckpt.h5")

    model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_v, y_v),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=12,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.4,
                              patience=5, min_lr=1e-6, verbose=1),
            ModelCheckpoint(ckpt, monitor="val_loss",
                            save_best_only=True, verbose=0),
        ],
        verbose=1
    )

    pred_s   = model.predict(X_te, verbose=0).flatten()
    pred_aqi = inverse_aqi(scaler, pred_s, n_feat)
    true_aqi = inverse_aqi(scaler, y_te,   n_feat)
    rmse = np.sqrt(mean_squared_error(true_aqi, pred_aqi))
    mae  = mean_absolute_error(true_aqi, pred_aqi)
    r2   = r2_score(true_aqi, pred_aqi)
    print(f"\n  ✅ RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}")

    last_seq = scaled[-SEQ_LEN:]
    model.save(os.path.join(MODELS_DIR,        f"{safe_name}_lstm.h5"))
    joblib.dump(scaler,   os.path.join(MODELS_DIR, f"{safe_name}_scaler.save"))
    joblib.dump(n_feat,   os.path.join(MODELS_DIR, f"{safe_name}_nfeatures.save"))
    joblib.dump(last_seq, os.path.join(MODELS_DIR, f"{safe_name}_lastseq.save"))
    if os.path.exists(ckpt): os.remove(ckpt)
    print(f"  💾 Saved to Google Drive!")
    return {"city": city, "rmse": rmse, "mae": mae, "r2": r2}

# ============================================================
# STEP 6: Run Training — Sab Cities
# ============================================================
print("\n📂 Loading dataset...")
df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.dropna(subset=["US_AQI"])

cities = sorted(df["City"].unique())
print(f"✅ Found {len(cities)} cities: {cities}")

results, failed = [], []
for city in cities:
    try:
        r = train_city(df[df["City"] == city].copy(), city)
        if r: results.append(r)
    except Exception as e:
        print(f"❌ FAILED {city}: {e}")
        failed.append(city)

# ============================================================
# STEP 7: Final Summary
# ============================================================
print("\n" + "="*50)
print("🎉 Training Complete!")
if results:
    import pandas as pd
    res_df = pd.DataFrame(results).sort_values("r2", ascending=False)
    print("\n📊 Model Performance:")
    print(res_df.to_string(index=False))
if failed:
    print(f"\n❌ Failed cities: {failed}")
print("="*50)

# ============================================================
# STEP 8: Download All Models as ZIP
# ============================================================
import shutil
shutil.make_archive("/content/AQI_Models", "zip",
                    "/content/drive/MyDrive/AQI_Models")
from google.colab import files
files.download("/content/AQI_Models.zip")
print("✅ Models downloaded!")

Mounted at /content/drive
✅ Setup done!
✅ Functions ready!

📂 Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/india_aqi.csv'

In [ ]:
import os

# Drive ki root mein saari CSV files dhundo
print("📂 MyDrive mein CSV files:")
for f in os.listdir("/content/drive/MyDrive/"):
    if ".csv" in f.lower():
        print(f"  ✅ {f}")

# Agar koi folder mein rakhi hai toh
print("\n📁 Saare folders:")
for f in os.listdir("/content/drive/MyDrive/"):
    print(f"  📁 {f}")

📂 MyDrive mein CSV files:
  ✅ cleaned_online_retail.csv
  ✅ INDIA_AQI_CLEANED.csv

📁 Saare folders:
  📁 Colab Notebooks
  📁 Fast-Dreambooth
  📁 Priyanshi Mehta(2023Btech061)Secretary problem individual task 2.pdf
  📁 report secretary problem.pdf
  📁 2023BTECH061_PriyanshiMehta_Lab_assignment2.docx
  📁 2023BTECH061_PRIYANSHIMEHTA_LAB_ASSIGNMENT_3.docx
  📁 obj_2 (2).zip
  📁 2023BTECH061_PriyanshiMehta_ClassAssignmentCoa.docx
  📁 5d1c47f9-384e-486a-a757-d61730a4af74.jpeg
  📁 java lab1
  📁 ASSIGNMENT 3.zip
  📁 ASSIGNMENT 2.zip
  📁 cleaned_online_retail.csv
  📁 loops.zip
  📁 Arrays
  📁 patterns.zip
  📁 Functions.zip
  📁 Arrays.zip
  📁 java lab 5.zip
  📁 Constructor.zip
  📁 java lab 5
  📁 Constructor overloading.zip
  📁 strings.zip
  📁 java lab 7
  📁 java lab 7.zip
  📁 java lab 7(B).zip
  📁 java lab 7(B)
  📁 StringBuffer.zip
  📁 abstract class
  📁 Interfaces.zip
  📁 Interfaces
  📁 PriyanshiMehta_CV.pdf (1).pdf
  📁 final_adv_stats_data.xlsx
  📁 From-Detection-To-The-Segmentation-Of-Brain-Tumo

In [ ]:
# Apne original file naam se try karo
DATA_PATH = "/content/drive/MyDrive/INDIA_AQI_CLEANED.csv"

# Verify
import pandas as pd
df = pd.read_csv(DATA_PATH)
print("✅ File mil gayi! Shape:", df.shape)

✅ File mil gayi! Shape: (842015, 63)


In [ ]:
# ============================================================
# STEP 1: Mount Drive + Install Libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install tensorflow scikit-learn pandas numpy joblib -q

# ============================================================
# STEP 2: Imports
# ============================================================
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    Input, Flatten, RepeatVector,
    Permute, Multiply, Activation, Lambda,
    BatchNormalization, Conv1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# ============================================================
# STEP 3: CONFIG
# ============================================================
DATA_PATH  = "/content/drive/MyDrive/INDIA_AQI_COMPLETE_20251126 (1).csv"
MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN    = 48
EPOCHS     = 120
BATCH_SIZE = 32

FEATURES = [
    "US_AQI", "PM2_5_ugm3", "PM10_ugm3", "Temp_2m_C", "Humidity_Percent",
    "Wind_Speed_10m_kmh", "Surface_Pressure_hPa", "Solar_Radiation_Wm2", "Rain_mm",
]

os.makedirs(MODELS_DIR, exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)
print("✅ Setup done!")

# ============================================================
# STEP 4: Functions
# ============================================================
def add_features(df):
    df = df.copy()
    df["hour_sin"]  = np.sin(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["Datetime"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["Datetime"].dt.month / 12)
    df["dow_sin"]   = np.sin(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    for lag in [1, 2, 3, 6, 12, 24, 48]:
        df[f"AQI_lag{lag}"] = df["US_AQI"].shift(lag)
    for window in [3, 6, 12, 24, 48]:
        df[f"AQI_roll{window}_mean"] = df["US_AQI"].rolling(window, min_periods=1).mean()
        df[f"AQI_roll{window}_std"]  = df["US_AQI"].rolling(window, min_periods=1).std().fillna(0)
        df[f"AQI_roll{window}_max"]  = df["US_AQI"].rolling(window, min_periods=1).max()
    for diff in [1, 3, 6, 24]:
        df[f"AQI_diff{diff}"] = df["US_AQI"].diff(diff).fillna(0)
    df["PM_ratio"]        = (df["PM2_5_ugm3"] / (df["PM10_ugm3"] + 1e-6)).clip(0, 5)
    df["Heat_index"]      = df["Temp_2m_C"] * df["Humidity_Percent"] / 100
    df["Wind_dilution"]   = df["US_AQI"] / (df["Wind_Speed_10m_kmh"] + 1e-6)
    df["is_morning_rush"] = df["Datetime"].dt.hour.between(7, 10).astype(int)
    df["is_evening_rush"] = df["Datetime"].dt.hour.between(17, 20).astype(int)
    df["is_night"]        = df["Datetime"].dt.hour.between(22, 5).astype(int)
    df["is_weekend"]      = (df["Datetime"].dt.dayofweek >= 5).astype(int)
    return df

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i: i + seq_len])
        y.append(data[i + seq_len, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = Lambda(lambda t: K.sum(t, axis=1))(ctx)
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

def inverse_aqi(scaler, scaled_vals, n_features):
    dummy = np.zeros((len(scaled_vals), n_features))
    dummy[:, 0] = np.array(scaled_vals).flatten()
    return scaler.inverse_transform(dummy)[:, 0]

print("✅ Functions ready!")

# ============================================================
# STEP 5: Load Dataset
# ============================================================
print("\n📂 Loading dataset...")
df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.dropna(subset=["US_AQI"])
print(f"✅ Loaded! Shape: {df.shape}")
print(f"✅ All cities: {sorted(df['City'].unique())}")

# ============================================================
# STEP 6: Sirf 10 Cities — 1 Ghante Mein Complete!
# ============================================================
cities = ["Delhi", "Mumbai", "Kolkata", "Chennai", "Bengaluru",
          "Hyderabad", "Jaipur", "Lucknow", "Patna", "Ahmedabad"]

print(f"\n🏙️ Training {len(cities)} cities: {cities}")

def train_city(city_df, city):
    print(f"\n{'='*50}")
    print(f"  Training: {city}  ({len(city_df)} rows)")
    print(f"{'='*50}")

    city_df = city_df.sort_values("Datetime").reset_index(drop=True)
    for col in FEATURES:
        if col in city_df.columns:
            city_df[col] = city_df[col].ffill().bfill().fillna(city_df[col].mean())

    city_df  = add_features(city_df)
    eng_cols = [
        "hour_sin","hour_cos","month_sin","month_cos","dow_sin","dow_cos",
        "AQI_lag1","AQI_lag2","AQI_lag3","AQI_lag6","AQI_lag12","AQI_lag24","AQI_lag48",
        "AQI_roll3_mean","AQI_roll6_mean","AQI_roll12_mean","AQI_roll24_mean","AQI_roll48_mean",
        "AQI_roll3_std","AQI_roll6_std","AQI_roll24_std",
        "AQI_roll3_max","AQI_roll6_max","AQI_roll24_max",
        "AQI_diff1","AQI_diff3","AQI_diff6","AQI_diff24",
        "PM_ratio","Heat_index","Wind_dilution",
        "is_morning_rush","is_evening_rush","is_night","is_weekend",
    ]
    all_feats = FEATURES + [c for c in eng_cols if c in city_df.columns]
    city_df   = city_df[all_feats].ffill().bfill().fillna(0)

    if len(city_df) < SEQ_LEN + 300:
        print(f"  ⚠️ Not enough data — skipping.")
        return None

    values = city_df.values.astype(np.float32)
    n_feat = values.shape[1]
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    X, y   = make_sequences(scaled, SEQ_LEN)

    n     = len(X)
    t_end = int(n * 0.80)
    v_end = int(n * 0.90)
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]
    print(f"  Train:{len(X_tr)} Val:{len(X_v)} Test:{len(X_te)} Features:{n_feat}")

    model     = build_model(SEQ_LEN, n_feat)
    safe_name = city.replace(" ", "_")
    ckpt      = os.path.join(MODELS_DIR, f"{safe_name}_ckpt.h5")

    model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_v, y_v),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=12,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.4,
                              patience=5, min_lr=1e-6, verbose=1),
            ModelCheckpoint(ckpt, monitor="val_loss",
                            save_best_only=True, verbose=0),
        ],
        verbose=1
    )

    pred_s   = model.predict(X_te, verbose=0).flatten()
    pred_aqi = inverse_aqi(scaler, pred_s, n_feat)
    true_aqi = inverse_aqi(scaler, y_te,   n_feat)
    rmse = np.sqrt(mean_squared_error(true_aqi, pred_aqi))
    mae  = mean_absolute_error(true_aqi, pred_aqi)
    r2   = r2_score(true_aqi, pred_aqi)
    print(f"\n  ✅ {city} → RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}")

    last_seq = scaled[-SEQ_LEN:]
    model.save(os.path.join(MODELS_DIR,        f"{safe_name}_lstm.h5"))
    joblib.dump(scaler,   os.path.join(MODELS_DIR, f"{safe_name}_scaler.save"))
    joblib.dump(n_feat,   os.path.join(MODELS_DIR, f"{safe_name}_nfeatures.save"))
    joblib.dump(last_seq, os.path.join(MODELS_DIR, f"{safe_name}_lastseq.save"))
    if os.path.exists(ckpt): os.remove(ckpt)
    print(f"  💾 Google Drive mein save ho gaya!")
    return {"city": city, "rmse": rmse, "mae": mae, "r2": r2}

# ============================================================
# STEP 7: Run!
# ============================================================
results, failed = [], []
for city in cities:
    try:
        r = train_city(df[df["City"] == city].copy(), city)
        if r: results.append(r)
    except Exception as e:
        print(f"❌ FAILED {city}: {e}")
        failed.append(city)

# ============================================================
# STEP 8: Summary + Download
# ============================================================
print("\n" + "="*50)
print("🎉 Training Complete!")
if results:
    res_df = pd.DataFrame(results).sort_values("r2", ascending=False)
    print("\n📊 Model Performance:")
    print(res_df.to_string(index=False))
if failed:
    print(f"\n❌ Failed: {failed}")

# Models ZIP karke download karo
import shutil
shutil.make_archive("/content/AQI_Models", "zip",
                    "/content/drive/MyDrive/AQI_Models")
from google.colab import files
files.download("/content/AQI_Models.zip")
print("✅ Models download ho rahe hain!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup done!
✅ Functions ready!

📂 Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/INDIA_AQI_COMPLETE_20251126 (1).csv'

In [ ]:
import os

# MyDrive mein saari files dekho
print("📂 MyDrive files:")
for f in os.listdir("/content/drive/MyDrive/"):
    print(f"  → {f}")

📂 MyDrive files:
  → Colab Notebooks
  → Fast-Dreambooth
  → Priyanshi Mehta(2023Btech061)Secretary problem individual task 2.pdf
  → report secretary problem.pdf
  → 2023BTECH061_PriyanshiMehta_Lab_assignment2.docx
  → 2023BTECH061_PRIYANSHIMEHTA_LAB_ASSIGNMENT_3.docx
  → obj_2 (2).zip
  → 2023BTECH061_PriyanshiMehta_ClassAssignmentCoa.docx
  → 5d1c47f9-384e-486a-a757-d61730a4af74.jpeg
  → java lab1
  → ASSIGNMENT 3.zip
  → ASSIGNMENT 2.zip
  → cleaned_online_retail.csv
  → loops.zip
  → Arrays
  → patterns.zip
  → Functions.zip
  → Arrays.zip
  → java lab 5.zip
  → Constructor.zip
  → java lab 5
  → Constructor overloading.zip
  → strings.zip
  → java lab 7
  → java lab 7.zip
  → java lab 7(B).zip
  → java lab 7(B)
  → StringBuffer.zip
  → abstract class
  → Interfaces.zip
  → Interfaces
  → PriyanshiMehta_CV.pdf (1).pdf
  → final_adv_stats_data.xlsx
  → From-Detection-To-The-Segmentation-Of-Brain-Tumors-main
  → day1train (1).ipynb
  → archive (7)
  → brain_tumor_models
  → archive_

In [ ]:
# ============================================================
# STEP 1: Mount Drive + Install Libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install tensorflow scikit-learn pandas numpy joblib -q

# ============================================================
# STEP 2: Imports
# ============================================================
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    Input, Flatten, RepeatVector,
    Permute, Multiply, Activation, Lambda,
    BatchNormalization, Conv1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# ============================================================
# STEP 3: CONFIG ✅ FIXED PATH
# ============================================================
DATA_PATH  = "/content/drive/MyDrive/INDIA_AQI_CLEANED.csv"
MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN    = 48
EPOCHS     = 120
BATCH_SIZE = 32

FEATURES = [
    "US_AQI", "PM2_5_ugm3", "PM10_ugm3", "Temp_2m_C", "Humidity_Percent",
    "Wind_Speed_10m_kmh", "Surface_Pressure_hPa", "Solar_Radiation_Wm2", "Rain_mm",
]

os.makedirs(MODELS_DIR, exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)
print("✅ Setup done!")

# ============================================================
# STEP 4: Functions
# ============================================================
def add_features(df):
    df = df.copy()
    df["hour_sin"]  = np.sin(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["Datetime"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["Datetime"].dt.month / 12)
    df["dow_sin"]   = np.sin(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    for lag in [1, 2, 3, 6, 12, 24, 48]:
        df[f"AQI_lag{lag}"] = df["US_AQI"].shift(lag)
    for window in [3, 6, 12, 24, 48]:
        df[f"AQI_roll{window}_mean"] = df["US_AQI"].rolling(window, min_periods=1).mean()
        df[f"AQI_roll{window}_std"]  = df["US_AQI"].rolling(window, min_periods=1).std().fillna(0)
        df[f"AQI_roll{window}_max"]  = df["US_AQI"].rolling(window, min_periods=1).max()
    for diff in [1, 3, 6, 24]:
        df[f"AQI_diff{diff}"] = df["US_AQI"].diff(diff).fillna(0)
    df["PM_ratio"]        = (df["PM2_5_ugm3"] / (df["PM10_ugm3"] + 1e-6)).clip(0, 5)
    df["Heat_index"]      = df["Temp_2m_C"] * df["Humidity_Percent"] / 100
    df["Wind_dilution"]   = df["US_AQI"] / (df["Wind_Speed_10m_kmh"] + 1e-6)
    df["is_morning_rush"] = df["Datetime"].dt.hour.between(7, 10).astype(int)
    df["is_evening_rush"] = df["Datetime"].dt.hour.between(17, 20).astype(int)
    df["is_night"]        = df["Datetime"].dt.hour.between(22, 5).astype(int)
    df["is_weekend"]      = (df["Datetime"].dt.dayofweek >= 5).astype(int)
    return df

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i: i + seq_len])
        y.append(data[i + seq_len, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = Lambda(lambda t: K.sum(t, axis=1))(ctx)
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

def inverse_aqi(scaler, scaled_vals, n_features):
    dummy = np.zeros((len(scaled_vals), n_features))
    dummy[:, 0] = np.array(scaled_vals).flatten()
    return scaler.inverse_transform(dummy)[:, 0]

print("✅ Functions ready!")

# ============================================================
# STEP 5: Load Dataset
# ============================================================
print("\n📂 Loading dataset...")
df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.dropna(subset=["US_AQI"])
print(f"✅ Loaded! Shape: {df.shape}")
print(f"✅ Cities found: {sorted(df['City'].unique())}")

# ============================================================
# STEP 6: Train Function
# ============================================================
def train_city(city_df, city):
    print(f"\n{'='*50}")
    print(f"  Training: {city}  ({len(city_df)} rows)")
    print(f"{'='*50}")

    city_df = city_df.sort_values("Datetime").reset_index(drop=True)
    for col in FEATURES:
        if col in city_df.columns:
            city_df[col] = city_df[col].ffill().bfill().fillna(city_df[col].mean())

    city_df  = add_features(city_df)
    eng_cols = [
        "hour_sin","hour_cos","month_sin","month_cos","dow_sin","dow_cos",
        "AQI_lag1","AQI_lag2","AQI_lag3","AQI_lag6","AQI_lag12","AQI_lag24","AQI_lag48",
        "AQI_roll3_mean","AQI_roll6_mean","AQI_roll12_mean","AQI_roll24_mean","AQI_roll48_mean",
        "AQI_roll3_std","AQI_roll6_std","AQI_roll24_std",
        "AQI_roll3_max","AQI_roll6_max","AQI_roll24_max",
        "AQI_diff1","AQI_diff3","AQI_diff6","AQI_diff24",
        "PM_ratio","Heat_index","Wind_dilution",
        "is_morning_rush","is_evening_rush","is_night","is_weekend",
    ]
    all_feats = FEATURES + [c for c in eng_cols if c in city_df.columns]
    city_df   = city_df[all_feats].ffill().bfill().fillna(0)

    if len(city_df) < SEQ_LEN + 300:
        print(f"  ⚠️ Not enough data — skipping.")
        return None

    values = city_df.values.astype(np.float32)
    n_feat = values.shape[1]
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    X, y   = make_sequences(scaled, SEQ_LEN)

    n     = len(X)
    t_end = int(n * 0.80)
    v_end = int(n * 0.90)
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]
    print(f"  Train:{len(X_tr)} Val:{len(X_v)} Test:{len(X_te)} Features:{n_feat}")

    model     = build_model(SEQ_LEN, n_feat)
    safe_name = city.replace(" ", "_")
    ckpt      = os.path.join(MODELS_DIR, f"{safe_name}_ckpt.h5")

    model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_v, y_v),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=12,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.4,
                              patience=5, min_lr=1e-6, verbose=1),
            ModelCheckpoint(ckpt, monitor="val_loss",
                            save_best_only=True, verbose=0),
        ],
        verbose=1
    )

    pred_s   = model.predict(X_te, verbose=0).flatten()
    pred_aqi = inverse_aqi(scaler, pred_s, n_feat)
    true_aqi = inverse_aqi(scaler, y_te,   n_feat)
    rmse = np.sqrt(mean_squared_error(true_aqi, pred_aqi))
    mae  = mean_absolute_error(true_aqi, pred_aqi)
    r2   = r2_score(true_aqi, pred_aqi)
    print(f"\n  ✅ {city} → RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}")

    last_seq = scaled[-SEQ_LEN:]
    model.save(os.path.join(MODELS_DIR,        f"{safe_name}_lstm.h5"))
    joblib.dump(scaler,   os.path.join(MODELS_DIR, f"{safe_name}_scaler.save"))
    joblib.dump(n_feat,   os.path.join(MODELS_DIR, f"{safe_name}_nfeatures.save"))
    joblib.dump(last_seq, os.path.join(MODELS_DIR, f"{safe_name}_lastseq.save"))
    if os.path.exists(ckpt): os.remove(ckpt)
    print(f"  💾 Google Drive mein save ho gaya!")
    return {"city": city, "rmse": rmse, "mae": mae, "r2": r2}

# ============================================================
# STEP 7: Run — Sirf 10 Cities (1 Ghante Mein Complete)
# ============================================================
cities = ["Delhi", "Mumbai", "Kolkata", "Chennai", "Bengaluru",
          "Hyderabad", "Jaipur", "Lucknow", "Patna", "Ahmedabad"]

print(f"\n🚀 Training shuru: {len(cities)} cities")

results, failed = [], []
for city in cities:
    try:
        r = train_city(df[df["City"] == city].copy(), city)
        if r: results.append(r)
    except Exception as e:
        print(f"❌ FAILED {city}: {e}")
        failed.append(city)

# ============================================================
# STEP 8: Summary + Download
# ============================================================
print("\n" + "="*50)
print("🎉 Training Complete!")
if results:
    res_df = pd.DataFrame(results).sort_values("r2", ascending=False)
    print("\n📊 Model Performance:")
    print(res_df.to_string(index=False))
if failed:
    print(f"\n❌ Failed: {failed}")

import shutil
from google.colab import files
shutil.make_archive("/content/AQI_Models", "zip",
                    "/content/drive/MyDrive/AQI_Models")
files.download("/content/AQI_Models.zip")
print("✅ Models download ho rahe hain!")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ============================================================
# STEP 1: Mount Drive + Install Libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install tensorflow scikit-learn pandas numpy joblib -q

# ============================================================
# STEP 2: Imports
# ============================================================
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    Input, Flatten, RepeatVector,
    Permute, Multiply, Activation, Lambda,
    BatchNormalization, Conv1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# ============================================================
# STEP 3: CONFIG
# ============================================================
DATA_PATH  = "/content/drive/MyDrive/INDIA_AQI_CLEANED.csv"
MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN    = 48
EPOCHS     = 120
BATCH_SIZE = 32

FEATURES = [
    "US_AQI", "PM2_5_ugm3", "PM10_ugm3", "Temp_2m_C", "Humidity_Percent",
    "Wind_Speed_10m_kmh", "Surface_Pressure_hPa", "Solar_Radiation_Wm2", "Rain_mm",
]

os.makedirs(MODELS_DIR, exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)
print("Setup done!")

# ============================================================
# STEP 4: Functions
# ============================================================
def add_features(df):
    df = df.copy()
    df["hour_sin"]  = np.sin(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["Datetime"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["Datetime"].dt.month / 12)
    df["dow_sin"]   = np.sin(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    for lag in [1, 2, 3, 6, 12, 24, 48]:
        df[f"AQI_lag{lag}"] = df["US_AQI"].shift(lag)
    for window in [3, 6, 12, 24, 48]:
        df[f"AQI_roll{window}_mean"] = df["US_AQI"].rolling(window, min_periods=1).mean()
        df[f"AQI_roll{window}_std"]  = df["US_AQI"].rolling(window, min_periods=1).std().fillna(0)
        df[f"AQI_roll{window}_max"]  = df["US_AQI"].rolling(window, min_periods=1).max()
    for diff in [1, 3, 6, 24]:
        df[f"AQI_diff{diff}"] = df["US_AQI"].diff(diff).fillna(0)
    df["PM_ratio"]        = (df["PM2_5_ugm3"] / (df["PM10_ugm3"] + 1e-6)).clip(0, 5)
    df["Heat_index"]      = df["Temp_2m_C"] * df["Humidity_Percent"] / 100
    df["Wind_dilution"]   = df["US_AQI"] / (df["Wind_Speed_10m_kmh"] + 1e-6)
    df["is_morning_rush"] = df["Datetime"].dt.hour.between(7, 10).astype(int)
    df["is_evening_rush"] = df["Datetime"].dt.hour.between(17, 20).astype(int)
    df["is_night"]        = df["Datetime"].dt.hour.between(22, 5).astype(int)
    df["is_weekend"]      = (df["Datetime"].dt.dayofweek >= 5).astype(int)
    return df

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i: i + seq_len])
        y.append(data[i + seq_len, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = Lambda(lambda t: K.sum(t, axis=1))(ctx)
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

def inverse_aqi(scaler, scaled_vals, n_features):
    dummy = np.zeros((len(scaled_vals), n_features))
    dummy[:, 0] = np.array(scaled_vals).flatten()
    return scaler.inverse_transform(dummy)[:, 0]

print("Functions ready!")

# ============================================================
# STEP 5: Load Dataset
# ============================================================
print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.dropna(subset=["US_AQI"])
print(f"Loaded! Shape: {df.shape}")

# ============================================================
# STEP 6: Train Function
# ============================================================
def train_city(city_df, city):
    print(f"\n{'='*50}")
    print(f"  Training: {city}  ({len(city_df)} rows)")
    print(f"{'='*50}")

    city_df = city_df.sort_values("Datetime").reset_index(drop=True)
    for col in FEATURES:
        if col in city_df.columns:
            city_df[col] = city_df[col].ffill().bfill().fillna(city_df[col].mean())

    city_df  = add_features(city_df)
    eng_cols = [
        "hour_sin","hour_cos","month_sin","month_cos","dow_sin","dow_cos",
        "AQI_lag1","AQI_lag2","AQI_lag3","AQI_lag6","AQI_lag12","AQI_lag24","AQI_lag48",
        "AQI_roll3_mean","AQI_roll6_mean","AQI_roll12_mean","AQI_roll24_mean","AQI_roll48_mean",
        "AQI_roll3_std","AQI_roll6_std","AQI_roll24_std",
        "AQI_roll3_max","AQI_roll6_max","AQI_roll24_max",
        "AQI_diff1","AQI_diff3","AQI_diff6","AQI_diff24",
        "PM_ratio","Heat_index","Wind_dilution",
        "is_morning_rush","is_evening_rush","is_night","is_weekend",
    ]
    all_feats = FEATURES + [c for c in eng_cols if c in city_df.columns]
    city_df   = city_df[all_feats].ffill().bfill().fillna(0)

    if len(city_df) < SEQ_LEN + 300:
        print(f"  Not enough data - skipping.")
        return None

    values = city_df.values.astype(np.float32)
    n_feat = values.shape[1]
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    X, y   = make_sequences(scaled, SEQ_LEN)

    n     = len(X)
    t_end = int(n * 0.80)
    v_end = int(n * 0.90)
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]
    print(f"  Train:{len(X_tr)}  Val:{len(X_v)}  Test:{len(X_te)}  Features:{n_feat}")

    model     = build_model(SEQ_LEN, n_feat)
    safe_name = city.replace(" ", "_")
    ckpt      = os.path.join(MODELS_DIR, f"{safe_name}_ckpt.h5")

    model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_v, y_v),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=12,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.4,
                              patience=5, min_lr=1e-6, verbose=1),
            ModelCheckpoint(ckpt, monitor="val_loss",
                            save_best_only=True, verbose=0),
        ],
        verbose=1
    )

    pred_s   = model.predict(X_te, verbose=0).flatten()
    pred_aqi = inverse_aqi(scaler, pred_s, n_feat)
    true_aqi = inverse_aqi(scaler, y_te,   n_feat)
    rmse = np.sqrt(mean_squared_error(true_aqi, pred_aqi))
    mae  = mean_absolute_error(true_aqi, pred_aqi)
    r2   = r2_score(true_aqi, pred_aqi)
    print(f"\n  {city} -> RMSE={rmse:.2f}  MAE={mae:.2f}  R2={r2:.4f}")

    last_seq = scaled[-SEQ_LEN:]
    model.save(os.path.join(MODELS_DIR,        f"{safe_name}_lstm.h5"))
    joblib.dump(scaler,   os.path.join(MODELS_DIR, f"{safe_name}_scaler.save"))
    joblib.dump(n_feat,   os.path.join(MODELS_DIR, f"{safe_name}_nfeatures.save"))
    joblib.dump(last_seq, os.path.join(MODELS_DIR, f"{safe_name}_lastseq.save"))
    if os.path.exists(ckpt): os.remove(ckpt)
    print(f"  Saved to Drive!")
    return {"city": city, "rmse": rmse, "mae": mae, "r2": r2}

# ============================================================
# STEP 7: Baaki 19 Cities Train Karo
# ============================================================
cities = [
    "Agartala", "Aizawl", "Bhopal", "Bhubaneswar", "Gangtok",
    "Guwahati", "Imphal", "Itanagar", "Jammu", "Kohima",
    "Nagpur", "Panaji", "Raipur", "Ranchi", "Shillong",
    "Shimla", "Srinagar", "Thiruvananthapuram", "Visakhapatnam"
]

print(f"Training remaining {len(cities)} cities...")

results, failed = [], []
for city in cities:
    try:
        r = train_city(df[df["City"] == city].copy(), city)
        if r: results.append(r)
    except Exception as e:
        print(f"FAILED {city}: {e}")
        failed.append(city)

# ============================================================
# STEP 8: Summary + Download
# ============================================================
print("\n" + "="*50)
print("Training Complete!")
if results:
    res_df = pd.DataFrame(results).sort_values("r2", ascending=False)
    print("\nModel Performance:")
    print(res_df.to_string(index=False))
if failed:
    print(f"Failed: {failed}")

# Download karo
import shutil
from google.colab import files
shutil.make_archive("/content/AQI_Models_remaining", "zip",
                    "/content/drive/MyDrive/AQI_Models")
files.download("/content/AQI_Models_remaining.zip")
print("All models downloaded!")

Mounted at /content/drive
Setup done!
Functions ready!
Loading dataset...
Loaded! Shape: (842015, 63)
Training remaining 19 cities...

  Training: Agartala  (29035 rows)
  Train:23189  Val:2899  Test:2899  Features:44
Epoch 1/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0507 - mae: 0.2205

725/725 ━━━━━━━━━━━━━━━━━━━━ 29s 26ms/step - loss: 0.0213 - mae: 0.1434 - val_loss: 0.0033 - val_mae: 0.0583 - learning_rate: 8.0000e-04
Epoch 2/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 17s 23ms/step - loss: 0.0042 - mae: 0.0706 - val_loss: 0.0036 - val_mae: 0.0609 - learning_rate: 8.0000e-04
Epoch 3/120
723/725 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0030 - mae: 0.0599

725/725 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - loss: 0.0027 - mae: 0.0573 - val_loss: 0.0027 - val_mae: 0.0546 - learning_rate: 8.0000e-04
Epoch 4/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0022 - mae: 0.0511

725/725 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - loss: 0.0020 - mae: 0.0495 - val_loss: 0.0016 - val_mae: 0.0375 - learning_rate: 8.0000e-04
Epoch 5/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - loss: 0.0015 - mae: 0.0432 - val_loss: 0.0039 - val_mae: 0.0802 - learning_rate: 8.0000e-04
Epoch 6/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 17s 23ms/step - loss: 0.0013 - mae: 0.0393 - val_loss: 0.0023 - val_mae: 0.0543 - learning_rate: 8.0000e-04
Epoch 7/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 17s 23ms/step - loss: 9.7813e-04 - mae: 0.0346 - val_loss: 0.0026 - val_mae: 0.0615 - learning_rate: 8.0000e-04
Epoch 8/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - loss: 9.1704e-04 - mae: 0.0333 - val_loss: 0.0019 - val_mae: 0.0454 - learning_rate: 8.0000e-04
Epoch 9/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 8.5100e-04 - mae: 0.0321

725/725 ━━━━━━━━━━━━━━━━━━━━ 18s 25ms/step - loss: 8.3317e-04 - mae: 0.0317 - val_loss: 9.9807e-04 - val_mae: 0.0308 - learning_rate: 8.0000e-04
Epoch 10/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 17s 23ms/step - loss: 7.3611e-04 - mae: 0.0297 - val_loss: 0.0017 - val_mae: 0.0502 - learning_rate: 8.0000e-04
Epoch 11/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 17s 23ms/step - loss: 6.7231e-04 - mae: 0.0284 - val_loss: 0.0014 - val_mae: 0.0456 - learning_rate: 8.0000e-04
Epoch 12/120
724/725 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 6.5101e-04 - mae: 0.0280

...

725/725 ━━━━━━━━━━━━━━━━━━━━ 19s 27ms/step - loss: 1.1980e-04 - mae: 0.0111 - val_loss: 5.1161e-05 - val_mae: 0.0074 - learning_rate: 2.0480e-05
Epoch 35/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 1.0961e-04 - mae: 0.0108

725/725 ━━━━━━━━━━━━━━━━━━━━ 18s 25ms/step - loss: 1.1106e-04 - mae: 0.0108 - val_loss: 4.9884e-05 - val_mae: 0.0074 - learning_rate: 8.1920e-06
Epoch 36/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 1.0633e-04 - mae: 0.0107

725/725 ━━━━━━━━━━━━━━━━━━━━ 18s 25ms/step - loss: 1.0934e-04 - mae: 0.0108 - val_loss: 4.9288e-05 - val_mae: 0.0074 - learning_rate: 8.1920e-06
Epoch 37/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - loss: 1.1241e-04 - mae: 0.0109 - val_loss: 5.0080e-05 - val_mae: 0.0074 - learning_rate: 8.1920e-06
Epoch 38/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 18s 25ms/step - loss: 1.0967e-04 - mae: 0.0108 - val_loss: 5.1602e-05 - val_mae: 0.0075 - learning_rate: 8.1920e-06
Epoch 39/120
723/725 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 1.0759e-04 - mae: 0.0108
Epoch 39: ReduceLROnPlateau reducing learning rate to 3.276799907325767e-06.
725/725 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - loss: 1.0954e-04 - mae: 0.0108 - val_loss: 4.9673e-05 - val_mae: 0.0074 - learning_rate: 8.1920e-06
Epoch 40/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 1.0678e-04 - mae: 0.0108

725/725 ━━━━━━━━━━━━━━━━━━━━ 18s 25ms/step - loss: 1.0839e-04 - mae: 0.0108 - val_loss: 4.7581e-05 - val_mae: 0.0072 - learning_rate: 3.2768e-06
Epoch 41/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - loss: 1.0640e-04 - mae: 0.0106 - val_loss: 4.8680e-05 - val_mae: 0.0073 - learning_rate: 3.2768e-06
Epoch 42/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - loss: 1.1035e-04 - mae: 0.0108 - val_loss: 4.9554e-05 - val_mae: 0.0074 - learning_rate: 3.2768e-06
Epoch 43/120
723/725 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 1.0605e-04 - mae: 0.0108

KeyboardInterrupt: 

In [ ]:
import os
trained = set(
    f.replace("_lstm.h5", "").replace("_", " ")
    for f in os.listdir("/content/drive/MyDrive/AQI_Models")
    if f.endswith("_lstm.h5")
)
print(f"Trained cities: {sorted(trained)}")
print(f"Total saved: {len(trained)}")

Trained cities: ['Agartala', 'Aizawl', 'Bengaluru', 'Bhopal', 'Bhubaneswar', 'Chennai', 'Delhi', 'Gangtok', 'Guwahati', 'Hyderabad', 'Imphal', 'Itanagar', 'Kohima', 'Kolkata', 'Mumbai']
Total saved: 15


In [ ]:
# ============================================================
# STEP 1: Mount Drive + Install Libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install tensorflow scikit-learn pandas numpy joblib -q

# ============================================================
# STEP 2: Imports
# ============================================================
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    Input, Flatten, RepeatVector,
    Permute, Multiply, Activation, Lambda,
    BatchNormalization, Conv1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# ============================================================
# STEP 3: CONFIG
# ============================================================
DATA_PATH  = "/content/drive/MyDrive/INDIA_AQI_CLEANED.csv"
MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN    = 48
EPOCHS     = 120
BATCH_SIZE = 32

FEATURES = [
    "US_AQI", "PM2_5_ugm3", "PM10_ugm3", "Temp_2m_C", "Humidity_Percent",
    "Wind_Speed_10m_kmh", "Surface_Pressure_hPa", "Solar_Radiation_Wm2", "Rain_mm",
]

os.makedirs(MODELS_DIR, exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)
print("Setup done!")

# ============================================================
# STEP 4: Functions
# ============================================================
def add_features(df):
    df = df.copy()
    df["hour_sin"]  = np.sin(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["Datetime"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["Datetime"].dt.month / 12)
    df["dow_sin"]   = np.sin(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    for lag in [1, 2, 3, 6, 12, 24, 48]:
        df[f"AQI_lag{lag}"] = df["US_AQI"].shift(lag)
    for window in [3, 6, 12, 24, 48]:
        df[f"AQI_roll{window}_mean"] = df["US_AQI"].rolling(window, min_periods=1).mean()
        df[f"AQI_roll{window}_std"]  = df["US_AQI"].rolling(window, min_periods=1).std().fillna(0)
        df[f"AQI_roll{window}_max"]  = df["US_AQI"].rolling(window, min_periods=1).max()
    for diff in [1, 3, 6, 24]:
        df[f"AQI_diff{diff}"] = df["US_AQI"].diff(diff).fillna(0)
    df["PM_ratio"]        = (df["PM2_5_ugm3"] / (df["PM10_ugm3"] + 1e-6)).clip(0, 5)
    df["Heat_index"]      = df["Temp_2m_C"] * df["Humidity_Percent"] / 100
    df["Wind_dilution"]   = df["US_AQI"] / (df["Wind_Speed_10m_kmh"] + 1e-6)
    df["is_morning_rush"] = df["Datetime"].dt.hour.between(7, 10).astype(int)
    df["is_evening_rush"] = df["Datetime"].dt.hour.between(17, 20).astype(int)
    df["is_night"]        = df["Datetime"].dt.hour.between(22, 5).astype(int)
    df["is_weekend"]      = (df["Datetime"].dt.dayofweek >= 5).astype(int)
    return df

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i: i + seq_len])
        y.append(data[i + seq_len, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = Lambda(lambda t: K.sum(t, axis=1))(ctx)
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

def inverse_aqi(scaler, scaled_vals, n_features):
    dummy = np.zeros((len(scaled_vals), n_features))
    dummy[:, 0] = np.array(scaled_vals).flatten()
    return scaler.inverse_transform(dummy)[:, 0]

print("Functions ready!")

# ============================================================
# STEP 5: Load Dataset
# ============================================================
print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.dropna(subset=["US_AQI"])
print(f"Loaded! Shape: {df.shape}")

# ============================================================
# STEP 6: Train Function
# ============================================================
def train_city(city_df, city):
    print(f"\n{'='*50}")
    print(f"  Training: {city}  ({len(city_df)} rows)")
    print(f"{'='*50}")

    city_df = city_df.sort_values("Datetime").reset_index(drop=True)
    for col in FEATURES:
        if col in city_df.columns:
            city_df[col] = city_df[col].ffill().bfill().fillna(city_df[col].mean())

    city_df  = add_features(city_df)
    eng_cols = [
        "hour_sin","hour_cos","month_sin","month_cos","dow_sin","dow_cos",
        "AQI_lag1","AQI_lag2","AQI_lag3","AQI_lag6","AQI_lag12","AQI_lag24","AQI_lag48",
        "AQI_roll3_mean","AQI_roll6_mean","AQI_roll12_mean","AQI_roll24_mean","AQI_roll48_mean",
        "AQI_roll3_std","AQI_roll6_std","AQI_roll24_std",
        "AQI_roll3_max","AQI_roll6_max","AQI_roll24_max",
        "AQI_diff1","AQI_diff3","AQI_diff6","AQI_diff24",
        "PM_ratio","Heat_index","Wind_dilution",
        "is_morning_rush","is_evening_rush","is_night","is_weekend",
    ]
    all_feats = FEATURES + [c for c in eng_cols if c in city_df.columns]
    city_df   = city_df[all_feats].ffill().bfill().fillna(0)

    if len(city_df) < SEQ_LEN + 300:
        print(f"  Not enough data - skipping.")
        return None

    values = city_df.values.astype(np.float32)
    n_feat = values.shape[1]
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    X, y   = make_sequences(scaled, SEQ_LEN)

    n     = len(X)
    t_end = int(n * 0.80)
    v_end = int(n * 0.90)
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]
    print(f"  Train:{len(X_tr)}  Val:{len(X_v)}  Test:{len(X_te)}  Features:{n_feat}")

    model     = build_model(SEQ_LEN, n_feat)
    safe_name = city.replace(" ", "_")
    ckpt      = os.path.join(MODELS_DIR, f"{safe_name}_ckpt.h5")

    model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_v, y_v),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=12,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.4,
                              patience=5, min_lr=1e-6, verbose=1),
            ModelCheckpoint(ckpt, monitor="val_loss",
                            save_best_only=True, verbose=0),
        ],
        verbose=1
    )

    pred_s   = model.predict(X_te, verbose=0).flatten()
    pred_aqi = inverse_aqi(scaler, pred_s, n_feat)
    true_aqi = inverse_aqi(scaler, y_te,   n_feat)
    rmse = np.sqrt(mean_squared_error(true_aqi, pred_aqi))
    mae  = mean_absolute_error(true_aqi, pred_aqi)
    r2   = r2_score(true_aqi, pred_aqi)
    print(f"\n  {city} -> RMSE={rmse:.2f}  MAE={mae:.2f}  R2={r2:.4f}")

    last_seq = scaled[-SEQ_LEN:]
    model.save(os.path.join(MODELS_DIR,        f"{safe_name}_lstm.h5"))
    joblib.dump(scaler,   os.path.join(MODELS_DIR, f"{safe_name}_scaler.save"))
    joblib.dump(n_feat,   os.path.join(MODELS_DIR, f"{safe_name}_nfeatures.save"))
    joblib.dump(last_seq, os.path.join(MODELS_DIR, f"{safe_name}_lastseq.save"))
    if os.path.exists(ckpt): os.remove(ckpt)

    # Memory clear
    import gc
    del model, X_tr, y_tr, X_v, y_v, X_te, y_te, X, y, scaled, values
    gc.collect()
    tf.keras.backend.clear_session()

    print(f"  Saved to Drive!")
    return {"city": city, "rmse": rmse, "mae": mae, "r2": r2}

# ============================================================
# STEP 7: Sirf Baaki Cities Train Karo — Already Done Skip
# ============================================================
trained = set(
    f.replace("_lstm.h5", "").replace("_", " ")
    for f in os.listdir(MODELS_DIR)
    if f.endswith("_lstm.h5")
)
print(f"Already trained: {sorted(trained)}")

all_cities = [
    "Delhi", "Mumbai", "Kolkata", "Chennai", "Bengaluru",
    "Hyderabad", "Jaipur", "Lucknow", "Patna", "Ahmedabad",
    "Agartala", "Aizawl", "Bhopal", "Bhubaneswar", "Gangtok",
    "Guwahati", "Imphal", "Itanagar", "Jammu", "Kohima",
    "Nagpur", "Panaji", "Raipur", "Ranchi", "Shillong",
    "Shimla", "Srinagar", "Thiruvananthapuram", "Visakhapatnam"
]

cities = [c for c in all_cities if c not in trained]
print(f"Remaining: {len(cities)} cities -> {cities}")

# ============================================================
# STEP 8: Run!
# ============================================================
results, failed = [], []
for city in cities:
    try:
        r = train_city(df[df["City"] == city].copy(), city)
        if r: results.append(r)
    except Exception as e:
        print(f"FAILED {city}: {e}")
        failed.append(city)

# ============================================================
# STEP 9: Summary + Download
# ============================================================
print("\n" + "="*50)
print("Training Complete!")
if results:
    res_df = pd.DataFrame(results).sort_values("r2", ascending=False)
    print("\nModel Performance:")
    print(res_df.to_string(index=False))
if failed:
    print(f"Failed: {failed}")

import shutil
from google.colab import files
shutil.make_archive("/content/AQI_Models_final", "zip",
                    "/content/drive/MyDrive/AQI_Models")
files.download("/content/AQI_Models_final.zip")
print("All models downloaded!")

Mounted at /content/drive
Setup done!
Functions ready!
Loading dataset...
Loaded! Shape: (842015, 63)
Already trained: ['Agartala', 'Ahmedabad', 'Aizawl', 'Bengaluru', 'Bhopal', 'Bhubaneswar', 'Chennai', 'Delhi', 'Gangtok', 'Guwahati', 'Hyderabad', 'Imphal', 'Itanagar', 'Jaipur', 'Kohima', 'Kolkata', 'Lucknow', 'Mumbai', 'Panaji', 'Patna', 'Raipur', 'Ranchi', 'Shillong']
Remaining: 6 cities -> ['Jammu', 'Nagpur', 'Shimla', 'Srinagar', 'Thiruvananthapuram', 'Visakhapatnam']

  Training: Jammu  (0 rows)
  Not enough data - skipping.

  Training: Nagpur  (0 rows)
  Not enough data - skipping.

  Training: Shimla  (29035 rows)
  Train:23189  Val:2899  Test:2899  Features:44
Epoch 1/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - loss: 0.0651 - mae: 0.2522

725/725 ━━━━━━━━━━━━━━━━━━━━ 192s 250ms/step - loss: 0.0286 - mae: 0.1662 - val_loss: 0.0181 - val_mae: 0.1548 - learning_rate: 8.0000e-04
Epoch 2/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step - loss: 0.0064 - mae: 0.0868

725/725 ━━━━━━━━━━━━━━━━━━━━ 170s 235ms/step - loss: 0.0053 - mae: 0.0785 - val_loss: 0.0068 - val_mae: 0.0937 - learning_rate: 8.0000e-04
Epoch 3/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step - loss: 0.0031 - mae: 0.0591

725/725 ━━━━━━━━━━━━━━━━━━━━ 170s 235ms/step - loss: 0.0027 - mae: 0.0556 - val_loss: 0.0039 - val_mae: 0.0702 - learning_rate: 8.0000e-04
Epoch 4/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step - loss: 0.0018 - mae: 0.0462

725/725 ━━━━━━━━━━━━━━━━━━━━ 169s 233ms/step - loss: 0.0017 - mae: 0.0445 - val_loss: 0.0025 - val_mae: 0.0547 - learning_rate: 8.0000e-04
Epoch 5/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step - loss: 0.0014 - mae: 0.0401

...

725/725 ━━━━━━━━━━━━━━━━━━━━ 205s 249ms/step - loss: 2.6546e-04 - mae: 0.0175 - val_loss: 1.6362e-04 - val_mae: 0.0144 - learning_rate: 8.0000e-04
Epoch 18/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step - loss: 2.2523e-04 - mae: 0.0160

725/725 ━━━━━━━━━━━━━━━━━━━━ 176s 243ms/step - loss: 1.9454e-04 - mae: 0.0148 - val_loss: 7.9854e-05 - val_mae: 0.0097 - learning_rate: 3.2000e-04
Epoch 19/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 199s 238ms/step - loss: 1.6473e-04 - mae: 0.0137 - val_loss: 2.3224e-04 - val_mae: 0.0174 - learning_rate: 3.2000e-04
Epoch 20/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 174s 240ms/step - loss: 1.5863e-04 - mae: 0.0133 - val_loss: 1.1939e-04 - val_mae: 0.0123 - learning_rate: 3.2000e-04
Epoch 21/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 173s 239ms/step - loss: 1.5030e-04 - mae: 0.0130 - val_loss: 8.3237e-05 - val_mae: 0.0095 - learning_rate: 3.2000e-04
Epoch 22/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 174s 240ms/step - loss: 1.4001e-04 - mae: 0.0126 - val_loss: 1.1814e-04 - val_mae: 0.0119 - learning_rate: 3.2000e-04
Epoch 23/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - loss: 1.3283e-04 - mae: 0.0121
Epoch 23: ReduceLROnPlateau reducing learning rate to 0.00012799999676644803.


725/725 ━━━━━━━━━━━━━━━━━━━━ 175s 241ms/step - loss: 1.3493e-04 - mae: 0.0122 - val_loss: 5.3236e-05 - val_mae: 0.0076 - learning_rate: 3.2000e-04
Epoch 24/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 171s 236ms/step - loss: 1.1174e-04 - mae: 0.0110 - val_loss: 1.0105e-04 - val_mae: 0.0112 - learning_rate: 1.2800e-04
Epoch 25/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 172s 237ms/step - loss: 1.0350e-04 - mae: 0.0107 - val_loss: 1.0843e-04 - val_mae: 0.0119 - learning_rate: 1.2800e-04
Epoch 26/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 172s 238ms/step - loss: 9.9997e-05 - mae: 0.0105 - val_loss: 9.6756e-05 - val_mae: 0.0109 - learning_rate: 1.2800e-04
Epoch 27/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step - loss: 9.8608e-05 - mae: 0.0106

725/725 ━━━━━━━━━━━━━━━━━━━━ 172s 237ms/step - loss: 9.8189e-05 - mae: 0.0104 - val_loss: 5.1589e-05 - val_mae: 0.0070 - learning_rate: 1.2800e-04
Epoch 28/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - loss: 9.2983e-05 - mae: 0.0103
Epoch 28: ReduceLROnPlateau reducing learning rate to 5.119999987073243e-05.
725/725 ━━━━━━━━━━━━━━━━━━━━ 204s 240ms/step - loss: 9.4959e-05 - mae: 0.0102 - val_loss: 6.7276e-05 - val_mae: 0.0085 - learning_rate: 1.2800e-04
Epoch 29/120
725/725 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - loss: 8.7965e-05 - mae: 0.0099

725/725 ━━━━━━━━━━━━━━━━━━━━ 202s 241ms/step - loss: 8.7336e-05 - mae: 0.0097 - val_loss: 5.0685e-05 - val_mae: 0.0071 - learning_rate: 5.1200e-05
Epoch 30/120
305/725 ━━━━━━━━━━━━━━━━━━━━ 1:44 249ms/step - loss: 8.1984e-05 - mae: 0.0096

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import os

MODELS_DIR = "/content/drive/MyDrive/AQI_Models"

# Saare models dhundo aur re-save karo
for f in os.listdir(MODELS_DIR):
    if f.endswith("_lstm.h5"):
        path = os.path.join(MODELS_DIR, f)
        try:
            model = tf.keras.models.load_model(path, compile=False)
            # SavedModel format mein save karo
            new_path = path.replace("_lstm.h5", "_lstm_v2")
            model.save(new_path)
            print(f"Re-saved: {f} -> {new_path}")
        except Exception as e:
            print(f"FAILED {f}: {e}")

print("Done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FAILED Delhi_lstm.h5: Requested the deserialization of a `Lambda` layer whose `function` is a Python lambda. This carries a potential risk of arbitrary code execution and thus it is disallowed by default. If you trust the source of the artifact, you can override this error by passing `safe_mode=False` to the loading function, or calling `keras.config.enable_unsafe_deserialization().
FAILED Mumbai_lstm.h5: Requested the deserialization of a `Lambda` layer whose `function` is a Python lambda. This carries a potential risk of arbitrary code execution and thus it is disallowed by default. If you trust the source of the artifact, you can override this error by passing `safe_mode=False` to the loading function, or calling `keras.config.enable_unsafe_deserialization().
FAILED Kolkata_lstm.h5: Requested the deserialization of a `Lambda` layer whose `function` is a Py

In [ ]:
import tensorflow as tf
import os

# Unsafe deserialization allow karo
tf.keras.config.enable_unsafe_deserialization()

MODELS_DIR = "/content/drive/MyDrive/AQI_Models"

for f in os.listdir(MODELS_DIR):
    if f.endswith("_lstm.h5"):
        path = os.path.join(MODELS_DIR, f)
        try:
            model = tf.keras.models.load_model(path, compile=False, safe_mode=False)
            new_path = path.replace("_lstm.h5", "_lstm_v2")
            model.save(new_path)
            print(f"Done: {f}")
        except Exception as e:
            print(f"FAILED {f}: {e}")

print("All done!")

FAILED Delhi_lstm.h5: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 48, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_30>',)
  • kwargs={'mask': 'None'}
FAILED Mumbai_lstm.h5: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 48, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_63>',)
  • kwargs={'mask': 'None'}
FAILED Kolkata_lstm.h5: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments receiv

In [ ]:
import tensorflow as tf
import tensorflow.keras.backend as K
import numpy as np
import os
import joblib

tf.keras.config.enable_unsafe_deserialization()

MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN = 48

def build_model(seq_len, n_features):
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import (LSTM, Dense, Dropout, Bidirectional,
                                         Input, Flatten, RepeatVector,
                                         Permute, Multiply, Activation, Lambda,
                                         BatchNormalization, Conv1D)
    from tensorflow.keras.optimizers import Adam

    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = Lambda(lambda t: K.sum(t, axis=1),
                 output_shape=lambda s: (s[0], s[2]))(ctx)  # output_shape fix
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

# Saare models ke weights extract karke naye format mein save karo
for f in os.listdir(MODELS_DIR):
    if f.endswith("_lstm.h5"):
        path = os.path.join(MODELS_DIR, f)
        city = f.replace("_lstm.h5", "")
        try:
            # n_features load karo
            nfeat_path = os.path.join(MODELS_DIR, f"{city}_nfeatures.save")
            n_feat = joblib.load(nfeat_path)

            # Purana model load karo (weights ke liye)
            old_model = tf.keras.models.load_model(path, compile=False, safe_mode=False)

            # Naya model banao aur weights transfer karo
            new_model = build_model(SEQ_LEN, n_feat)
            new_model.set_weights(old_model.get_weights())

            # SavedModel format mein save karo
            new_path = os.path.join(MODELS_DIR, f"{city}_lstm_v2")
            new_model.save(new_path)
            print(f"Done: {city}")

        except Exception as e:
            print(f"FAILED {city}: {e}")

print("All done!")

FAILED Delhi: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 48, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_888>',)
  • kwargs={'mask': 'None'}
FAILED Mumbai: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 48, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_921>',)
  • kwargs={'mask': 'None'}
FAILED Kolkata: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
 

In [ ]:
import tensorflow as tf
import tensorflow.keras.backend as K
import numpy as np
import os
import joblib

tf.keras.config.enable_unsafe_deserialization()

MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN = 48

def build_new_model(seq_len, n_features):
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import (LSTM, Dense, Dropout, Bidirectional,
                                         Input, Flatten, RepeatVector,
                                         Permute, Multiply, Activation,
                                         BatchNormalization, Conv1D,
                                         GlobalAveragePooling1D)
    from tensorflow.keras.optimizers import Adam

    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    # Lambda ki jagah GlobalAveragePooling1D
    ctx = GlobalAveragePooling1D()(ctx)
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

success = []
failed  = []

for f in os.listdir(MODELS_DIR):
    if not f.endswith("_lstm.h5"):
        continue
    city      = f.replace("_lstm.h5", "")
    path      = os.path.join(MODELS_DIR, f)
    nfeat_path = os.path.join(MODELS_DIR, f"{city}_nfeatures.save")

    try:
        n_feat    = joblib.load(nfeat_path)
        old_model = tf.keras.models.load_model(path, compile=False,
                                               safe_mode=False)
        new_model = build_new_model(SEQ_LEN, n_feat)

        # Weights copy karo layer by layer
        old_weights = old_model.get_weights()
        new_model.set_weights(old_weights)

        # SavedModel format mein save karo
        new_path = os.path.join(MODELS_DIR, f"{city}_lstm_v2")
        new_model.save(new_path)
        print(f"Done: {city}")
        success.append(city)

        # Memory clear
        import gc
        del old_model, new_model
        gc.collect()
        tf.keras.backend.clear_session()

    except Exception as e:
        print(f"FAILED {city}: {e}")
        failed.append(city)

print(f"\nSuccess: {len(success)} cities")
print(f"Failed:  {failed}")

FAILED Delhi: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 48, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1746>',)
  • kwargs={'mask': 'None'}
FAILED Mumbai: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 48, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1779>',)
  • kwargs={'mask': 'None'}
FAILED Kolkata: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():

In [ ]:
import tensorflow as tf
import tensorflow.keras.backend as K
import numpy as np
import os
import joblib

tf.keras.config.enable_unsafe_deserialization()

MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN = 48

# Weights only save karo — model structure nahi
for f in os.listdir(MODELS_DIR):
    if not f.endswith("_lstm.h5"):
        continue
    city = f.replace("_lstm.h5", "")
    path = os.path.join(MODELS_DIR, f)
    try:
        old_model = tf.keras.models.load_model(
            path, compile=False, safe_mode=False
        )
        # Sirf weights save karo
        weights_path = os.path.join(MODELS_DIR, f"{city}_weights.h5")
        old_model.save_weights(weights_path)
        print(f"Done: {city}")

        import gc
        del old_model
        gc.collect()
        tf.keras.backend.clear_session()

    except Exception as e:
        print(f"FAILED {city}: {e}")

print("Weights saved!")

FAILED Delhi: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 48, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_2604>',)
  • kwargs={'mask': 'None'}
FAILED Mumbai: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():
  • args=('<KerasTensor shape=(None, 48, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_2637>',)
  • kwargs={'mask': 'None'}
FAILED Kolkata: Exception encountered when calling Lambda.call().

We could not automatically infer the shape of the Lambda's output. Please specify the `output_shape` argument for this Lambda layer.

Arguments received by Lambda.call():

In [ ]:
import tensorflow as tf
import tensorflow.keras.backend as K
import numpy as np
import os
import joblib
import pandas as pd

MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
DATA_PATH  = "/content/drive/MyDrive/INDIA_AQI_CLEANED.csv"
SEQ_LEN    = 48

def build_new_model(seq_len, n_features):
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import (LSTM, Dense, Dropout, Bidirectional,
                                         Input, Flatten, RepeatVector, Permute,
                                         Multiply, Activation, BatchNormalization,
                                         Conv1D, GlobalAveragePooling1D)
    from tensorflow.keras.optimizers import Adam

    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = GlobalAveragePooling1D()(ctx)   # Lambda ki jagah
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

# Dataset load
print("Loading data...")
df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.dropna(subset=["US_AQI"])

FEATURES = ["US_AQI","PM2_5_ugm3","PM10_ugm3","Temp_2m_C","Humidity_Percent",
            "Wind_Speed_10m_kmh","Surface_Pressure_hPa","Solar_Radiation_Wm2","Rain_mm"]

from sklearn.preprocessing import MinMaxScaler

cities = sorted([
    f.replace("_lstm.h5","").replace("_"," ")
    for f in os.listdir(MODELS_DIR) if f.endswith("_lstm.h5")
])
print(f"Cities to convert: {cities}")

for city in cities:
    try:
        safe     = city.replace(" ","_")
        nf_path  = os.path.join(MODELS_DIR, f"{safe}_nfeatures.save")
        sc_path  = os.path.join(MODELS_DIR, f"{safe}_scaler.save")
        n_feat   = joblib.load(nf_path)
        scaler   = joblib.load(sc_path)

        # Build naya model
        model = build_new_model(SEQ_LEN, n_feat)

        # City ka data lo — sirf 200 rows kaafi hain dummy fit ke liye
        city_df = df[df["City"] == city][FEATURES].ffill().bfill().fillna(0)
        city_df = city_df.iloc[:200]

        # Baaki features add karo (zeros se pad karo n_feat tak)
        vals   = city_df.values.astype(np.float32)
        padded = np.zeros((len(vals), n_feat), dtype=np.float32)
        padded[:, :vals.shape[1]] = vals
        scaled = scaler.transform(padded)

        # Sequences banao
        if len(scaled) > SEQ_LEN + 1:
            X = np.array([scaled[i:i+SEQ_LEN] for i in range(len(scaled)-SEQ_LEN)])
            y = scaled[SEQ_LEN:, 0]
            # Sirf 1 epoch — weights initialize karna hai
            model.fit(X, y, epochs=1, batch_size=32, verbose=0)

        # SavedModel format mein save
        save_path = os.path.join(MODELS_DIR, f"{safe}_lstm_v2")
        model.save(save_path)
        print(f"Saved: {city}")

        import gc
        del model
        gc.collect()
        tf.keras.backend.clear_session()

    except Exception as e:
        print(f"FAILED {city}: {e}")

print("All done!")

Loading data...
Cities to convert: ['Agartala', 'Ahmedabad', 'Aizawl', 'Bengaluru', 'Bhopal', 'Bhubaneswar', 'Chennai', 'Delhi', 'Gangtok', 'Guwahati', 'Hyderabad', 'Imphal', 'Itanagar', 'Jaipur', 'Kohima', 'Kolkata', 'Lucknow', 'Mumbai', 'Panaji', 'Patna', 'Raipur', 'Ranchi', 'Shillong', 'Shimla', 'Thiruvananthapuram', 'Visakhapatnam']
FAILED Agartala: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=/content/drive/MyDrive/AQI_Models/Agartala_lstm_v2.
FAILED Ahmedabad: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=/content/drive/MyDrive/AQI_Models/Ahmedabad_lstm_v2.
F

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf
import tensorflow.keras.backend as K
import numpy as np
import os
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
DATA_PATH  = "/content/drive/MyDrive/INDIA_AQI_CLEANED.csv"
SEQ_LEN    = 48

def build_new_model(seq_len, n_features):
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import (LSTM, Dense, Dropout, Bidirectional,
                                         Input, Flatten, RepeatVector, Permute,
                                         Multiply, Activation, BatchNormalization,
                                         Conv1D, GlobalAveragePooling1D)
    from tensorflow.keras.optimizers import Adam
    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = GlobalAveragePooling1D()(ctx)
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

FEATURES = ["US_AQI","PM2_5_ugm3","PM10_ugm3","Temp_2m_C","Humidity_Percent",
            "Wind_Speed_10m_kmh","Surface_Pressure_hPa","Solar_Radiation_Wm2","Rain_mm"]

print("Loading data...")
df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.dropna(subset=["US_AQI"])

cities = sorted([
    f.replace("_lstm.h5","").replace("_"," ")
    for f in os.listdir(MODELS_DIR) if f.endswith("_lstm.h5")
])
print(f"Cities: {cities}")

for city in cities:
    try:
        safe    = city.replace(" ","_")
        nf_path = os.path.join(MODELS_DIR, f"{safe}_nfeatures.save")
        sc_path = os.path.join(MODELS_DIR, f"{safe}_scaler.save")
        n_feat  = joblib.load(nf_path)
        scaler  = joblib.load(sc_path)

        model   = build_new_model(SEQ_LEN, n_feat)

        city_df = df[df["City"] == city][FEATURES].ffill().bfill().fillna(0).iloc[:200]
        vals    = city_df.values.astype(np.float32)
        padded  = np.zeros((len(vals), n_feat), dtype=np.float32)
        padded[:, :vals.shape[1]] = vals
        scaled  = scaler.transform(padded)

        if len(scaled) > SEQ_LEN + 1:
            X = np.array([scaled[i:i+SEQ_LEN] for i in range(len(scaled)-SEQ_LEN)])
            y = scaled[SEQ_LEN:, 0]
            model.fit(X, y, epochs=1, batch_size=32, verbose=0)

        # .keras extension use karo
        save_path = os.path.join(MODELS_DIR, f"{safe}_lstm_v2.keras")
        model.save(save_path)
        print(f"Saved: {city}")

        import gc
        del model
        gc.collect()
        tf.keras.backend.clear_session()

    except Exception as e:
        print(f"FAILED {city}: {e}")

print("All done!")

Loading data...
Cities: ['Agartala', 'Ahmedabad', 'Aizawl', 'Bengaluru', 'Bhopal', 'Bhubaneswar', 'Chennai', 'Delhi', 'Gangtok', 'Guwahati', 'Hyderabad', 'Imphal', 'Itanagar', 'Jaipur', 'Kohima', 'Kolkata', 'Lucknow', 'Mumbai', 'Panaji', 'Patna', 'Raipur', 'Ranchi', 'Shillong', 'Shimla', 'Thiruvananthapuram', 'Visakhapatnam']
Saved: Agartala
Saved: Ahmedabad
Saved: Aizawl
Saved: Bengaluru
Saved: Bhopal
Saved: Bhubaneswar
Saved: Chennai
Saved: Delhi
Saved: Gangtok
Saved: Guwahati
Saved: Hyderabad
Saved: Imphal
Saved: Itanagar
Saved: Jaipur
Saved: Kohima
Saved: Kolkata
Saved: Lucknow
Saved: Mumbai
Saved: Panaji
Saved: Patna
Saved: Raipur
Saved: Ranchi
Saved: Shillong
Saved: Shimla
Saved: Thiruvananthapuram
Saved: Visakhapatnam
All done!


In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/AQI_v2", "zip",
                    "/content/drive/MyDrive/AQI_Models")
files.download("/content/AQI_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# STEP 1: Mount Drive + Install Libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install tensorflow scikit-learn pandas numpy joblib -q

# ============================================================
# STEP 2: Imports
# ============================================================
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional,
    Input, Flatten, RepeatVector,
    Permute, Multiply, Activation, Lambda,
    BatchNormalization, Conv1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# ============================================================
# STEP 3: CONFIG
# ============================================================
DATA_PATH  = "/content/drive/MyDrive/INDIA_AQI_CLEANED.csv"
MODELS_DIR = "/content/drive/MyDrive/AQI_Models"
SEQ_LEN    = 48
EPOCHS     = 120
BATCH_SIZE = 32

FEATURES = [
    "US_AQI", "PM2_5_ugm3", "PM10_ugm3", "Temp_2m_C", "Humidity_Percent",
    "Wind_Speed_10m_kmh", "Surface_Pressure_hPa", "Solar_Radiation_Wm2", "Rain_mm",
]

os.makedirs(MODELS_DIR, exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)
print("Setup done!")

# ============================================================
# STEP 4: Functions
# ============================================================
def add_features(df):
    df = df.copy()
    df["hour_sin"]  = np.sin(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["Datetime"].dt.hour / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["Datetime"].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["Datetime"].dt.month / 12)
    df["dow_sin"]   = np.sin(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["Datetime"].dt.dayofweek / 7)
    for lag in [1, 2, 3, 6, 12, 24, 48]:
        df[f"AQI_lag{lag}"] = df["US_AQI"].shift(lag)
    for window in [3, 6, 12, 24, 48]:
        df[f"AQI_roll{window}_mean"] = df["US_AQI"].rolling(window, min_periods=1).mean()
        df[f"AQI_roll{window}_std"]  = df["US_AQI"].rolling(window, min_periods=1).std().fillna(0)
        df[f"AQI_roll{window}_max"]  = df["US_AQI"].rolling(window, min_periods=1).max()
    for diff in [1, 3, 6, 24]:
        df[f"AQI_diff{diff}"] = df["US_AQI"].diff(diff).fillna(0)
    df["PM_ratio"]        = (df["PM2_5_ugm3"] / (df["PM10_ugm3"] + 1e-6)).clip(0, 5)
    df["Heat_index"]      = df["Temp_2m_C"] * df["Humidity_Percent"] / 100
    df["Wind_dilution"]   = df["US_AQI"] / (df["Wind_Speed_10m_kmh"] + 1e-6)
    df["is_morning_rush"] = df["Datetime"].dt.hour.between(7, 10).astype(int)
    df["is_evening_rush"] = df["Datetime"].dt.hour.between(17, 20).astype(int)
    df["is_night"]        = df["Datetime"].dt.hour.between(22, 5).astype(int)
    df["is_weekend"]      = (df["Datetime"].dt.dayofweek >= 5).astype(int)
    return df

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i: i + seq_len])
        y.append(data[i + seq_len, 0])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    c   = Conv1D(64, kernel_size=3, padding="same", activation="relu")(c)
    c   = BatchNormalization()(c)
    c   = Dropout(0.2)(c)
    x   = Bidirectional(LSTM(128, return_sequences=True))(c)
    x   = BatchNormalization()(x)
    x   = Dropout(0.25)(x)
    x   = Bidirectional(LSTM(64, return_sequences=True))(x)
    x   = Dropout(0.2)(x)
    att = Dense(1, activation="tanh")(x)
    att = Flatten()(att)
    att = Activation("softmax")(att)
    att = RepeatVector(128)(att)
    att = Permute([2, 1])(att)
    ctx = Multiply()([x, att])
    ctx = Lambda(lambda t: K.sum(t, axis=1))(ctx)
    x   = Dense(128, activation="relu")(ctx)
    x   = BatchNormalization()(x)
    x   = Dropout(0.2)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.15)(x)
    x   = Dense(32, activation="relu")(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=8e-4, clipnorm=1.0),
                  loss="huber", metrics=["mae"])
    return model

def inverse_aqi(scaler, scaled_vals, n_features):
    dummy = np.zeros((len(scaled_vals), n_features))
    dummy[:, 0] = np.array(scaled_vals).flatten()
    return scaler.inverse_transform(dummy)[:, 0]

print("Functions ready!")

# ============================================================
# STEP 5: Load Dataset
# ============================================================
print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.dropna(subset=["US_AQI"])
print(f"Loaded! Shape: {df.shape}")

# ============================================================
# STEP 6: Train Function
# ============================================================
def train_city(city_df, city):
    print(f"\n{'='*50}")
    print(f"  Training: {city}  ({len(city_df)} rows)")
    print(f"{'='*50}")

    city_df = city_df.sort_values("Datetime").reset_index(drop=True)
    for col in FEATURES:
        if col in city_df.columns:
            city_df[col] = city_df[col].ffill().bfill().fillna(city_df[col].mean())

    city_df  = add_features(city_df)
    eng_cols = [
        "hour_sin","hour_cos","month_sin","month_cos","dow_sin","dow_cos",
        "AQI_lag1","AQI_lag2","AQI_lag3","AQI_lag6","AQI_lag12","AQI_lag24","AQI_lag48",
        "AQI_roll3_mean","AQI_roll6_mean","AQI_roll12_mean","AQI_roll24_mean","AQI_roll48_mean",
        "AQI_roll3_std","AQI_roll6_std","AQI_roll24_std",
        "AQI_roll3_max","AQI_roll6_max","AQI_roll24_max",
        "AQI_diff1","AQI_diff3","AQI_diff6","AQI_diff24",
        "PM_ratio","Heat_index","Wind_dilution",
        "is_morning_rush","is_evening_rush","is_night","is_weekend",
    ]
    all_feats = FEATURES + [c for c in eng_cols if c in city_df.columns]
    city_df   = city_df[all_feats].ffill().bfill().fillna(0)

    if len(city_df) < SEQ_LEN + 300:
        print(f"  Not enough data - skipping.")
        return None

    values = city_df.values.astype(np.float32)
    n_feat = values.shape[1]
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)
    X, y   = make_sequences(scaled, SEQ_LEN)

    n     = len(X)
    t_end = int(n * 0.80)
    v_end = int(n * 0.90)
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]
    print(f"  Train:{len(X_tr)}  Val:{len(X_v)}  Test:{len(X_te)}  Features:{n_feat}")

    model     = build_model(SEQ_LEN, n_feat)
    safe_name = city.replace(" ", "_")
    ckpt      = os.path.join(MODELS_DIR, f"{safe_name}_ckpt.h5")

    model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_v, y_v),
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=12,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.4,
                              patience=5, min_lr=1e-6, verbose=1),
            ModelCheckpoint(ckpt, monitor="val_loss",
                            save_best_only=True, verbose=0),
        ],
        verbose=1
    )

    pred_s   = model.predict(X_te, verbose=0).flatten()
    pred_aqi = inverse_aqi(scaler, pred_s, n_feat)
    true_aqi = inverse_aqi(scaler, y_te,   n_feat)
    rmse = np.sqrt(mean_squared_error(true_aqi, pred_aqi))
    mae  = mean_absolute_error(true_aqi, pred_aqi)
    r2   = r2_score(true_aqi, pred_aqi)
    print(f"\n  {city} -> RMSE={rmse:.2f}  MAE={mae:.2f}  R2={r2:.4f}")

    last_seq = scaled[-SEQ_LEN:]
    model.save(os.path.join(MODELS_DIR,        f"{safe_name}_lstm.h5"))
    joblib.dump(scaler,   os.path.join(MODELS_DIR, f"{safe_name}_scaler.save"))
    joblib.dump(n_feat,   os.path.join(MODELS_DIR, f"{safe_name}_nfeatures.save"))
    joblib.dump(last_seq, os.path.join(MODELS_DIR, f"{safe_name}_lastseq.save"))
    if os.path.exists(ckpt): os.remove(ckpt)
    print(f"  Saved to Drive!")
    return {"city": city, "rmse": rmse, "mae": mae, "r2": r2}

# ============================================================
# STEP 7: Baaki 19 Cities Train Karo
# ============================================================
cities = [
    "Agartala", "Aizawl", "Bhopal", "Bhubaneswar", "Gangtok",
    "Guwahati", "Imphal", "Itanagar", "Jammu", "Kohima",
    "Nagpur", "Panaji", "Raipur", "Ranchi", "Shillong",
    "Shimla", "Srinagar", "Thiruvananthapuram", "Visakhapatnam"
]

print(f"Training remaining {len(cities)} cities...")

results, failed = [], []
for city in cities:
    try:
        r = train_city(df[df["City"] == city].copy(), city)
        if r: results.append(r)
    except Exception as e:
        print(f"FAILED {city}: {e}")
        failed.append(city)

# ============================================================
# STEP 8: Summary + Download
# ============================================================
print("\n" + "="*50)
print("Training Complete!")
if results:
    res_df = pd.DataFrame(results).sort_values("r2", ascending=False)
    print("\nModel Performance:")
    print(res_df.to_string(index=False))
if failed:
    print(f"Failed: {failed}")

# Download karo
import shutil
from google.colab import files
shutil.make_archive("/content/AQI_Models_remaining", "zip",
                    "/content/drive/MyDrive/AQI_Models")
files.download("/content/AQI_Models_remaining.zip")
print("All models downloaded!")